In [ ]:
# %%
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from calibration import expected_calibration_error
from common import DATASET_TO_LABEL, METHOD_TO_LABEL, get_color

plt.style.use(["science", "nature"])

DF = pd.read_parquet("dataframe/calibration.parquet")


def _plot_reliability_column(
    ax0,
    ax1,
    method,
    dataset,
    show_legend=False,
    show_annotations=False,
):
    bp, bf, bw = (
        DF[(DF.dataset == dataset) & (DF.method == method) & (DF.run_id == 0)][
            ["bin_probability", "bin_frequency", "bin_weight"]
        ]
        .to_numpy()
        .T
    )

    ece = expected_calibration_error(bp, bf, bw)
    color = get_color(method)

    bp = np.insert(bp, 0, 0)
    bf = np.insert(bf, 0, 0)
    bw = np.insert(bw, 0, 0)

    # Reliability curve
    ax0.plot([0, 1], [0, 1], "-", color="black", linewidth=0.8)
    x = np.linspace(0, 1, len(bp))
    ax0.plot(x, bf, ds="steps-pre", color=color, linewidth=1.2)

    ax0.set_xlim(0, 1)
    ax0.set_ylim(0, 1)
    ax0.set_xlabel("")
    ax0.set_ylabel(r"Accuracy $\operatorname{acc}\left(B_m\right)$")
    ax0.set_xticks(np.linspace(0, 1, 6))
    ax0.set_yticks(np.linspace(0, 1, 6))
    # ax0.minorticks_off()
    ax0.tick_params(axis="x", which="both", bottom=True, top=True, labelbottom=False)

    if show_annotations:
        ax0.text(
            0.25,
            0.75,
            r"$\leftarrow$ Unconfident",
            color="black",
            ha="center",
            va="center",
            rotation=-45,
            # path_effects=[withStroke(linewidth=2.0, foreground="white")],
        )
        ax0.text(
            0.75,
            0.25,
            r"Overconfident $\rightarrow$",
            color="black",
            ha="center",
            va="center",
            rotation=-45,
            # path_effects=[withStroke(linewidth=2.0, foreground="white")],
        )

    # Histogram of bin weights
    ax1.hist(
        x,
        weights=bw,
        bins=len(x),
        range=(0, 1),
        edgecolor="black",
        linewidth=0.5,
        color=color,
    )
    ax1.set_xlabel("")
    ax1.set_ylabel("Bin Weight\n$\\frac{|B_m|}{n}$")
    ax1.set_xlim(0, 1)
    ax1.set_xticks(np.linspace(0, 1, 6))
    # ax1.minorticks_off()
    ax1.tick_params(axis="x", which="both", bottom=True, top=True, labelbottom=True)

    avg_confidence = np.sum(bp * bw)
    avg_accuracy = np.sum(bf * bw)
    ax1.axvline(
        avg_confidence,
        color="black",
        linestyle="-",
        linewidth=0.9,
        label="Avg. Confidence",
    )
    ax1.axvline(
        avg_accuracy, color="black", linestyle="--", linewidth=0.9, label="Accuracy"
    )

    if show_legend:
        ax1.legend(
            frameon=False,
            loc="upper center",
            ncol=2,
            bbox_to_anchor=(0.5, -0.25),
        )

    ax0.set_title(f"{METHOD_TO_LABEL[method]}")
    ax0.text(
        0.5,
        0.96,
        f"ECE={ece * 100:.2f}",
        transform=ax0.transAxes,
        ha="center",
        va="top",
    )


def plot_reliability_diagrams_side_by_side(
    dataset,
    methods,
    figure_width_pt=397,
    aspect_ratio=1.9,
    show_suptitle=False,
):
    methods = list(methods)

    fig_width = figure_width_pt / 72.27
    fig_height = fig_width / aspect_ratio

    fig, axes = plt.subplots(
        2,
        len(methods),
        figsize=(fig_width, fig_height),
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.0},
        sharex="col",
        sharey="row",
    )

    if len(methods) == 1:
        axes = np.array(axes).reshape(2, 1)

    center_col = len(methods) // 2
    legend_col = center_col
    annotation_col = len(methods) // 2
    for col, method in enumerate(methods):
        _plot_reliability_column(
            axes[0, col],
            axes[1, col],
            method,
            dataset,
            show_legend=(col == legend_col),
            show_annotations=(col == annotation_col),
        )

    for col in range(1, len(methods)):
        axes[0, col].set_ylabel("")
        axes[1, col].set_ylabel("")
        # axes[0, col].tick_params(axis="y", which="both", labelleft=False)

    for col in range(len(methods)):
        axes[1, col].tick_params(
            axis="y", which="both", left=False, right=False, labelleft=False
        )

    axes[1, 0].set_xlabel(r"Confidence $\operatorname{conf}\left(B_m\right)$")

    axes[0, 0].yaxis.set_label_coords(-0.22, 0.5)
    # axes[1, 0].yaxis.set_label_coords(-0.22, 0.5)
    fig.subplots_adjust(bottom=0.20)

    if show_suptitle:
        fig.suptitle(f"{DATASET_TO_LABEL[dataset]} Reliability Diagrams", y=1.02)

    filename = Path("plots/reliability-diagram.pdf")
    fig.savefig(filename, bbox_inches="tight", pad_inches=0.02)
    fig.savefig(
        filename.with_suffix(".png"), dpi=300, bbox_inches="tight", pad_inches=0.02
    )
    print(f"Saved to {filename}")

In [ ]:
plot_reliability_diagrams_side_by_side(
    "cifar100",
    ["ball", "tball", "lora"],
    figure_width_pt=397,
    aspect_ratio=2.38,
    show_suptitle=False,
)